# 08 RQ7 User Trust and Usability Evaluation After Deployment

This notebook evaluates the trustworthiness, clarity, usability, and practical usefulness
of the deployed uncertainty-aware and explainable plant disease diagnosis system.

It performs:
- user-study CSV loading
- descriptive summary of participant ratings
- scenario-level response analysis
- workflow visualization
- export of tables and figures

Outputs:
- Table 13: user trust and usability summary
- Table 14: scenario-based user response patterns
- Figure 13: end-user workflow of the deployed system
- Figure 14: user-centered evaluation of trust, clarity, and usability
- ZIP archive of RQ7 outputs

In [4]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

In [5]:
# ----------------------------------------
# Section 2: Configuration
# ----------------------------------------

CONFIG = {
    # Update this path to your uploaded or attached Kaggle dataset CSV
    "user_study_csv": "/kaggle/input/datasets/thedataeng/study-submissiions/plant-disease-detector.study_submissions.csv",
    "output_root": "/kaggle/working/thesis_outputs/rq7_user_study",
}

OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print(f"User study CSV: {CONFIG['user_study_csv']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
User study CSV: /kaggle/input/datasets/thedataeng/study-submissiions/plant-disease-detector.study_submissions.csv
Output root: /kaggle/working/thesis_outputs/rq7_user_study


In [6]:
# ----------------------------------------
# Section 3: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

def percentage(part: float, total: float) -> float:
    """
    Safely calculate percentage values.
    """
    if total == 0:
        return 0.0

    return round((part / total) * 100, 2)

print('Done')

Done


In [7]:
# ----------------------------------------
# Section 4: Load user-study responses
# ----------------------------------------

csv_path = Path(CONFIG["user_study_csv"])
assert csv_path.exists(), f"Missing user study CSV: {csv_path}"

responses_raw = pd.read_csv(csv_path)

print("User-study CSV loaded successfully")
print(f"Raw rows/documents: {len(responses_raw)}")
print(f"Raw columns: {len(responses_raw.columns)}")

# MongoDB export uses participantId, not participant_id
required_columns = {
    "participantId",
    "orderId",
    "finalResponse.clearerVersion",
    "finalResponse.moreUsefulVersion",
    "finalResponse.saferVersion",
    "finalResponse.reuseScore",
}

missing_columns = required_columns - set(responses_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in user-study CSV: {missing_columns}")

print(f"Number of participants: {responses_raw['participantId'].nunique()}")

display(responses_raw.head())

User-study CSV loaded successfully
Raw rows/documents: 108
Raw columns: 44
Number of participants: 108


,_id,participantId,orderId,caseResponses[0].participantId,caseResponses[1].participantId,caseResponses[2].participantId,caseResponses[3].participantId,caseResponses[0].orderId,caseResponses[1].orderId,caseResponses[2].orderId,...,caseResponses[3].timestamp,finalResponse.participantId,finalResponse.orderId,finalResponse.clearerVersion,finalResponse.moreUsefulVersion,finalResponse.saferVersion,finalResponse.reuseScore,finalResponse.comment,finalResponse.timestamp,submittedAt
0,69fb434e3caec1c158479883,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,order_a,order_a,...,2026-05-06T13:32:57.101Z,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,full,full,full,5,NaN,2026-05-06T13:34:06.325Z,2026-05-06T13:34:06.325Z
1,69fc4c883caec1c158479884,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,order_a,order_a,...,2026-05-07T08:25:17.311Z,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,simple,simple,simple,4,NaN,2026-05-07T08:25:43.302Z,2026-05-07T08:25:43.302Z
2,69fc4cb73caec1c158479885,870d5847-4307-47db-9670-a183490956c0,order_a,870d5847-4307-47db-9670-a183490956c0,870d5847-4307-47db-9670-a183490956c0,870d5847-4307-47db-9670-a183490956c0,870d5847-4307-47db-9670-a183490956c0,order_a,order_a,order_a,...,2026-05-07T08:25:55.866Z,870d5847-4307-47db-9670-a183490956c0,order_a,full,full,full,4,NaN,2026-05-07T08:26:27.733Z,2026-05-07T08:26:27.733Z
3,69fc4cb93caec1c158479886,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,order_a,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,order_a,order_a,order_a,...,2026-05-07T08:25:57.504Z,07dae8cb-ce9f-4b89-b358-cac8e6f4a597,order_a,full,full,full,4,NaN,2026-05-07T08:26:28.419Z,2026-05-07T08:26:28.419Z
4,69fc4cc13caec1c158479887,e746c100-6482-4403-a591-d1bdbcd82e15,order_a,e746c100-6482-4403-a591-d1bdbcd82e15,e746c100-6482-4403-a591-d1bdbcd82e15,e746c100-6482-4403-a591-d1bdbcd82e15,e746c100-6482-4403-a591-d1bdbcd82e15,order_a,order_a,order_a,...,2026-05-07T08:25:46.555Z,e746c100-6482-4403-a591-d1bdbcd82e15,order_a,simple,full,full,3,NaN,2026-05-07T08:26:43.896Z,2026-05-07T08:26:43.897Z


In [8]:
# ----------------------------------------
# Section 5: Flatten case-level user-study responses
# ----------------------------------------

case_rows = []

for _, row in responses_raw.iterrows():

    participant_id = row["participantId"]
    order_id = row["orderId"]

    # Each participant has 4 case responses:
    # caseResponses[0], caseResponses[1], caseResponses[2], caseResponses[3]
    for i in range(4):

        prefix = f"caseResponses[{i}]"

        case_id_col = f"{prefix}.caseId"
        interface_col = f"{prefix}.interfaceType"
        action_col = f"{prefix}.selectedAction"
        clarity_col = f"{prefix}.clarityScore"
        trust_col = f"{prefix}.trustScore"

        # Skip if this case response is missing
        if case_id_col not in responses_raw.columns:
            continue

        case_rows.append({
            "participantId": participant_id,
            "orderId": order_id,
            "caseIndex": i,
            "caseId": row[case_id_col],
            "interfaceType": row[interface_col],
            "selectedAction": row[action_col],
            "clarityScore": row[clarity_col],
            "trustScore": row[trust_col],
        })


case_df = pd.DataFrame(case_rows)

# Convert scores to numeric
case_df["clarityScore"] = pd.to_numeric(case_df["clarityScore"], errors="coerce")
case_df["trustScore"] = pd.to_numeric(case_df["trustScore"], errors="coerce")

print("Case-level responses flattened successfully")
print(f"Number of participants: {case_df['participantId'].nunique()}")
print(f"Number of case-level responses: {len(case_df)}")

display(case_df.head(10))

Case-level responses flattened successfully
Number of participants: 108
Number of case-level responses: 432


,participantId,orderId,caseIndex,caseId,interfaceType,selectedAction,clarityScore,trustScore
0,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,0,simple_easy,simple,ACCEPT,5,5
1,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,1,simple_hard,simple,MONITOR,4,4
2,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,2,full_easy,full,ACCEPT,5,5
3,463ee6f2-61b7-41a6-b79c-6ec33e3e5e13,order_a,3,full_hard,full,RETAKE,4,4
4,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,0,simple_easy,simple,ACCEPT,5,4
5,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,1,simple_hard,simple,RETAKE,3,3
6,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,2,full_easy,full,MONITOR,4,4
7,1444dd26-33b3-423f-9f6a-e2b9c0282ce1,order_a,3,full_hard,full,RETAKE,2,1
8,870d5847-4307-47db-9670-a183490956c0,order_a,0,simple_easy,simple,MONITOR,4,4
9,870d5847-4307-47db-9670-a183490956c0,order_a,1,simple_hard,simple,RETAKE,4,4


In [9]:
# ----------------------------------------
# Section 6: Save Table 13 - Interface-level trust and clarity summary
# ----------------------------------------

table13_df = case_df.groupby("interfaceType").agg(
    MeanTrustScore=("trustScore", "mean"),
    MeanClarityScore=("clarityScore", "mean"),
    ResponseCount=("participantId", "count"),
).reset_index()

table13_df["MeanTrustScore"] = table13_df["MeanTrustScore"].apply(pretty_metric)
table13_df["MeanClarityScore"] = table13_df["MeanClarityScore"].apply(pretty_metric)

table13_df["interfaceType"] = table13_df["interfaceType"].replace({
    "simple": "Baseline Interface",
    "full": "Full Framework Interface",
})

save_table(table13_df, "Table_13_Interface_Level_User_Evaluation")

print("Table 13 saved successfully")
display(table13_df)

Saved table: /kaggle/working/thesis_outputs/rq7_user_study/tables/Table_13_Interface_Level_User_Evaluation.csv
Table 13 saved successfully


,interfaceType,MeanTrustScore,MeanClarityScore,ResponseCount
0,Full Framework Interface,3.6065,3.8148,216
1,Baseline Interface,3.0000,3.7870,216


In [10]:
# ----------------------------------------
# Section 7: Save Table 14 - Final participant preference summary
# ----------------------------------------

preference_questions = {
    "Clearer Interface": "finalResponse.clearerVersion",
    "More Useful Interface": "finalResponse.moreUsefulVersion",
    "Safer Interface": "finalResponse.saferVersion",
}

table14_rows = []

n_participants = responses_raw["participantId"].nunique()

for question_label, column_name in preference_questions.items():

    counts = responses_raw[column_name].value_counts(dropna=False)

    simple_count = counts.get("simple", 0)
    full_count = counts.get("full", 0)
    no_difference_count = counts.get("no_difference", 0)

    table14_rows.append({
        "Final Preference Question": question_label,

        
        "Baseline Interface (%)": percentage(simple_count, n_participants),

       
        "Full Framework Interface (%)": percentage(full_count, n_participants),

       
        "No Difference (%)": percentage(no_difference_count, n_participants),

        "Total Responses": int(n_participants),
    })


table14_df = pd.DataFrame(table14_rows)

save_table(table14_df, "Table_14_Final_Participant_Preferences")

print("Table 14 saved successfully")
display(table14_df)

Saved table: /kaggle/working/thesis_outputs/rq7_user_study/tables/Table_14_Final_Participant_Preferences.csv
Table 14 saved successfully


,Final Preference Question,Baseline Interface (%),Full Framework Interface (%),No Difference (%),Total Responses
0,Clearer Interface,23.15,75.00,1.85,108
1,More Useful Interface,13.89,84.26,1.85,108
2,Safer Interface,16.67,76.85,6.48,108


In [11]:
# ----------------------------------------
# Section 8: Figure 14 - Final participant preference comparison
# ----------------------------------------

clearer_counts = responses_raw["finalResponse.clearerVersion"].value_counts()
useful_counts = responses_raw["finalResponse.moreUsefulVersion"].value_counts()
safer_counts = responses_raw["finalResponse.saferVersion"].value_counts()

preference_labels = ["Simple", "Full", "No Difference"]

clearer_values = [
    clearer_counts.get("simple", 0),
    clearer_counts.get("full", 0),
    clearer_counts.get("no_difference", 0),
]

useful_values = [
    useful_counts.get("simple", 0),
    useful_counts.get("full", 0),
    useful_counts.get("no_difference", 0),
]

safer_values = [
    safer_counts.get("simple", 0),
    safer_counts.get("full", 0),
    safer_counts.get("no_difference", 0),
]

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(preference_labels))
width = 0.25

ax.bar(x - width, clearer_values, width, label="Clearer")
ax.bar(x, useful_values, width, label="More Useful")
ax.bar(x + width, safer_values, width, label="Safer")

ax.set_xticks(x)
ax.set_xticklabels(preference_labels)

ax.set_ylabel("Number of Participants")
ax.set_title("Figure 14. Final Participant Preference Comparison Between Interfaces")

ax.legend(frameon=True)
ax.grid(axis="y", alpha=0.25)

save_figure(fig, "Figure_14_Final_Participant_Preferences")

Saved figure: /kaggle/working/thesis_outputs/rq7_user_study/figures/Figure_14_Final_Participant_Preferences.pdf


In [12]:
# ----------------------------------------
# Section 9: Save RQ7 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq7_meta = {
    "num_participants": int(responses_raw["participantId"].nunique()),
    "num_raw_submissions": int(len(responses_raw)),
    "num_case_level_responses": int(len(case_df)),
    "case_ids": sorted(case_df["caseId"].dropna().unique().tolist()),
    "interface_types": sorted(case_df["interfaceType"].dropna().unique().tolist()),
    "case_level_evaluation_dimensions": [
        "trustScore",
        "clarityScore",
        "selectedAction",
    ],
    "final_response_dimensions": [
        "finalResponse.clearerVersion",
        "finalResponse.moreUsefulVersion",
        "finalResponse.saferVersion",
        "finalResponse.reuseScore",
    ],
    "analysis_notes": (
        "RQ7 analysis uses case-level responses for trust and clarity comparisons, "
        "and final participant responses for overall preferences regarding clarity, "
        "usefulness, safety, and reuse intention."
    ),
}

meta_path = meta_dir / "rq7_metadata.json"

with open(meta_path, "w") as f:
    json.dump(rq7_meta, f, indent=2)

print("RQ7 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ7 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq7_user_study/metadata/rq7_metadata.json


In [13]:
# ----------------------------------------
# Section 10: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "08_rq7_user_study_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("08_rq7_user_study notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/08_rq7_user_study_outputs.zip
08_rq7_user_study notebook completed successfully
